# RNN — Parte 4: Entrenamiento con PyTorch
### Dataset Jena Climate — Predicción de temperatura 24h

---

## ¿Qué hace este notebook?

Cargamos los datos preparados en la **Parte 1** y entrenamos dos modelos RNN:

| Modelo | Capas RNN | Descripción |
|---|---|---|
| `SimpleRNNModel` | 1 capa | RNN básica — punto de partida |
| `StackedRNNModel` | 2 capas | RNN apilada — más capacidad |

Al final comparamos ambos contra el **baseline naive** de la Parte 1.

## 🗺️ Flujo del notebook

```
Cargar datos  →  Dataset + DataLoader  →  Funciones de entrenamiento
     ↓
SimpleRNN  →  Entrenamiento  →  Evaluación  →  Visualización
     ↓
StackedRNN →  Entrenamiento  →  Evaluación  →  Visualización
     ↓
Tabla comparativa final (Naive vs Simple vs Stacked)
```


---
## Celda 1 — Importaciones y semilla


In [ ]:
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

# ✅ Semilla para reproducibilidad (misma que Parte 1)
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size']  = 11

print('✅ Librerías cargadas')
print(f'   PyTorch version : {torch.__version__}')
print(f'   CUDA disponible : {torch.cuda.is_available()}')

---
## Celda 2 — Montar Google Drive y cargar datos

Cargamos exactamente los archivos que guardó la **Parte 1**.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/Colab Notebooks/processed'

In [ ]:
# ── Cargar arrays ──────────────────────────────────────────────────────────────
raw_data    = np.load(os.path.join(path, 'raw_data.npy'))     # (N, 14) normalizado
temperature = np.load(os.path.join(path, 'temperature.npy'))  # (N,)    en °C real
train_idx   = np.load(os.path.join(path, 'train_idx.npy'))
val_idx     = np.load(os.path.join(path, 'val_idx.npy'))
test_idx    = np.load(os.path.join(path, 'test_idx.npy'))
scaler      = joblib.load(os.path.join(path, 'scaler.pkl'))
baseline_mae = np.load(os.path.join(path, 'baseline_mae.npy'))[0]

num_train_samples = len(train_idx)
num_val_samples   = len(val_idx)
num_test_samples  = len(test_idx)

# Media y std de la temperatura para desnormalizar predicciones
temp_mean = scaler.mean_[1]
temp_std  = scaler.scale_[1]

print('=' * 55)
print('📂 DATOS CARGADOS DESDE PARTE 1')
print('=' * 55)
print(f'  raw_data    : {raw_data.shape}    (normalizado)')
print(f'  temperature : {temperature.shape}  (°C real)')
print(f'  Train / Val / Test : {num_train_samples:,} / {num_val_samples:,} / {num_test_samples:,}')
print(f'  temp_mean = {temp_mean:.4f}°C   temp_std = {temp_std:.4f}°C')
print(f'  Baseline MAE (naive) = {baseline_mae:.4f}°C  ← meta a superar')

---
## Concepto: TimeseriesDataset — ¿Qué hace exactamente?

PyTorch necesita que los datos estén envueltos en un objeto `Dataset`.
Para series temporales, este dataset implementa la **ventana deslizante** que estudiamos en la Parte 1.

Cuando PyTorch pide la muestra número `idx`, el dataset:

1. Calcula el punto de inicio: `start = indices[idx]`
2. Selecciona los `sequence_length` pasos con el `sampling_rate`: `steps = [start, start+6, start+12, ..., start+714]`
3. Extrae X: los valores de todas las features en esos pasos → shape `(120, 14)`
4. Extrae y: la temperatura real 24h después → `temperature[start + delay]`
5. Devuelve el par `(X_tensor, y_tensor)`

```
Dataset[0]  →  X = raw_data[[0,6,12,...,714]],    y = temperature[858]
Dataset[1]  →  X = raw_data[[1,7,13,...,715]],    y = temperature[859]
Dataset[2]  →  X = raw_data[[2,8,14,...,716]],    y = temperature[860]
```


---
## Celda 3 — Hiperparámetros de ventana y TimeseriesDataset


In [ ]:
# ── Hiperparámetros de ventana (mismos que Parte 1) ───────────────────────────
sampling_rate   = 6    # 1 muestra por hora (cada 6 × 10min)
sequence_length = 120  # 120 horas = 5 días de historia
delay           = sampling_rate * (sequence_length + 24 - 1)  # = 858 pasos
batch_size      = 256

print(f'sampling_rate={sampling_rate}  sequence_length={sequence_length}  delay={delay}  batch_size={batch_size}')
print()

class TimeseriesDataset(Dataset):
    """
    Dataset de ventana deslizante para series temporales.

    Para cada índice devuelve:
      X : tensor (sequence_length, num_features)  ← historia submuestreada
      y : escalar float                           ← temperatura real 24h después
    """
    def __init__(self, data, targets, sequence_length, sampling_rate,
                 start_index, end_index):
        self.data            = data
        self.targets         = targets
        self.sequence_length = sequence_length
        self.sampling_rate   = sampling_rate
        # Índices de inicio válidos para este split
        self.indices         = np.arange(start_index, end_index)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        start = self.indices[idx]
        # Seleccionar pasos submuestreados: [start, start+6, ..., start+714]
        steps = np.arange(
            start,
            start + self.sequence_length * self.sampling_rate,
            self.sampling_rate
        )
        x = self.data[steps]                  # (120, 14)  — todas las features
        y = self.targets[start + delay]        # escalar    — temperatura en °C
        return (torch.tensor(x, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32))

    def __repr__(self):
        return (f'TimeseriesDataset('
                f'n_samples={len(self)}, '
                f'seq_len={self.sequence_length}, '
                f'features={self.data.shape[-1]})')

---
## Celda 4 — Crear Datasets y DataLoaders

>  El `end_index` del test usa `len(raw_data) - delay` para evitar que
> el índice `start + delay` se salga del array. Es el bug que tenía el original.


In [ ]:
train_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=0,
    end_index=num_train_samples,
)

val_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples,
    end_index=num_train_samples + num_val_samples,
)

# end_index = len(raw_data) - delay  → garantiza que start+delay no se pase
test_dataset = TimeseriesDataset(
    data=raw_data, targets=temperature,
    sequence_length=sequence_length, sampling_rate=sampling_rate,
    start_index=num_train_samples + num_val_samples,
    end_index=len(raw_data) - delay,
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)

print(train_dataset)
print(val_dataset)
print(test_dataset)
print()

# Sanity check — verificar shapes de un batch
X_batch, y_batch = next(iter(train_loader))
print(f'Batch X shape : {X_batch.shape}  ← (batch, sequence_length, num_features)')
print(f'Batch y shape : {y_batch.shape}  ← (batch,)')
print(f'y ejemplo     : {y_batch[:5].numpy()}  °C')

---
## Concepto: ¿Qué hace la RNN internamente?

La RNN procesa la secuencia **paso a paso**, de izquierda a derecha.
En cada paso `t` recibe el vector de features de esa hora y actualiza su **estado oculto** `h`:

```
h_t = tanh( Wxh · x_t  +  Whh · h_{t-1}  +  bh )
       ↑               ↑               ↑
  input actual    estado anterior    bias
```

Después de procesar los 120 pasos, tomamos el **último estado oculto** `h_119`
y lo pasamos por una capa lineal para obtener la predicción de temperatura:

```
ŷ = W_out · h_119 + b_out
```

### ¿Por qué solo el último estado?

Porque `h_119` ya contiene la información **acumulada** de todos los pasos anteriores.
Es el resumen de los 5 días de historia que necesitamos para predecir.

```
x[0]→[RNN]→h[0]→[RNN]→h[1]→ ... →[RNN]→h[119]→[Linear]→ ŷ
x[1]↗           x[2]↗                   x[119]↗
```


---
## Celda 5 — Definición de modelos

### Modelo 1 — SimpleRNN (1 capa)
### Modelo 2 — StackedRNN (2 capas apiladas)


In [ ]:
class SimpleRNNModel(nn.Module):
    """
    RNN de una sola capa.
    Arquitectura: RNN(14→hidden) → Linear(hidden→1)
    """
    def __init__(self, num_features, hidden_size=16):
        super().__init__()
        #  Bug corregido: 'hiesdden_size' → 'hidden_size'
        self.rnn  = nn.RNN(
            input_size=num_features,
            hidden_size=hidden_size,
            batch_first=True          # input: (batch, seq, features)
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, 120, 14)
        out, _ = self.rnn(x)          # out: (batch, 120, hidden_size)
        last   = out[:, -1, :]        # (batch, hidden_size) — último estado
        return self.head(last).squeeze(-1)  # (batch,)


class StackedRNNModel(nn.Module):
    """
    RNN apilada con 2 capas.
    La capa 2 recibe como entrada la secuencia de estados ocultos de la capa 1.
    Arquitectura: RNN×2(14→hidden) → Linear(hidden→1)
    """
    def __init__(self, num_features, hidden_size=16):
        super().__init__()
        self.rnn  = nn.RNN(
            input_size=num_features,
            hidden_size=hidden_size,
            num_layers=2,             # 2 capas apiladas
            batch_first=True,
            dropout=0.0               # se puede subir a 0.2–0.3 si hay overfitting
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :]).squeeze(-1)


# ── Verificar arquitecturas ───────────────────────────────────────────────────
num_features = raw_data.shape[-1]  # 14

for name, Model in [('SimpleRNN', SimpleRNNModel), ('StackedRNN', StackedRNNModel)]:
    m = Model(num_features=num_features)
    n_params = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f'{name:12s} — parámetros entrenables: {n_params:,}')

---
## Concepto: Callbacks de entrenamiento

Los callbacks son objetos que se ejecutan al final de cada época y toman decisiones automáticas:

| Callback | ¿Qué hace? |
|---|---|
| `ModelCheckpoint` | Guarda el modelo cuando el `val_mae` mejora |
| `EarlyStopping` | Para el entrenamiento si el `val_mae` no mejora en N épocas |
| `ReduceLROnPlateau` | Baja el learning rate si el `val_mae` se estanca |

Los tres trabajan juntos:
1. Si el modelo mejora → Checkpoint lo guarda
2. Si lleva K épocas sin mejorar → ReduceLR baja el LR para intentar salir del plateau
3. Si lleva N épocas sin mejorar → EarlyStopping detiene todo para no desperdiciar tiempo


---
## Celda 6 — Callbacks de entrenamiento


In [ ]:
class ModelCheckpoint:
    """Guarda el modelo cuando el metric monitorizado mejora."""
    def __init__(self, filepath, monitor='val_mae', mode='min', verbose=True):
        self.filepath = filepath
        self.monitor  = monitor
        self.verbose  = verbose
        self.best     = float('inf') if mode == 'min' else float('-inf')
        self.mode     = mode

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = value < self.best if self.mode == 'min' else value > self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f'    ✓ Mejor modelo guardado ({self.monitor}: {value:.4f}°C)')
        return improved


class EarlyStopping:
    """Para el entrenamiento si el metric no mejora en `patience` épocas."""
    def __init__(self, monitor='val_mae', patience=5, min_delta=1e-4, mode='min'):
        self.monitor     = monitor
        self.patience    = patience
        self.min_delta   = min_delta
        self.mode        = mode
        self.best        = float('inf') if mode == 'min' else float('-inf')
        self.counter     = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value    = metrics[self.monitor]
        improved = (value < self.best - self.min_delta if self.mode == 'min'
                    else value > self.best + self.min_delta)
        if improved:
            self.best    = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f'    ⏹ EarlyStopping activado (sin mejora en {self.patience} épocas)')
        return improved


class ReduceLROnPlateau:
    """Baja el learning rate cuando el metric se estanca."""
    def __init__(self, optimizer, monitor='val_mae', patience=3, factor=0.5):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, patience=patience, factor=factor, verbose=True
        )
        self.monitor = monitor

    def step(self, metrics, model=None):
        self.scheduler.step(metrics[self.monitor])


print('✅ Callbacks definidos: ModelCheckpoint | EarlyStopping | ReduceLROnPlateau')

---
## Concepto: Funciones de entrenamiento

### `run_epoch` — Una pasada completa por el dataset

En **modo entrenamiento** (`optimizer` presente):
1. Forward pass → calcula `ŷ`
2. Calcula loss MSE entre `ŷ` y `y`
3. `loss.backward()` → calcula gradientes
4. `optimizer.step()` → actualiza pesos

En **modo evaluación** (`optimizer=None`):
- Solo el forward pass, sin actualizar pesos
- `torch.no_grad()` desactiva el cálculo de gradientes → más rápido y usa menos memoria

### ¿Por qué MSE como loss pero reportamos MAE?

- **MSE** (Mean Squared Error): penaliza más los errores grandes → bueno para optimizar
- **MAE** (Mean Absolute Error): en °C → fácil de interpretar para humanos

Entrenamos con MSE, pero evaluamos con MAE para poder comparar contra el baseline.


---
## Celda 7 — Funciones de entrenamiento y evaluación


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """
    Una época completa de entrenamiento o evaluación.

    Args:
        model     : el modelo PyTorch
        loader    : DataLoader de train, val o test
        criterion : función de loss (MSELoss)
        optimizer : si None → modo eval (sin actualizar pesos)

    Returns:
        avg_loss : MSE promedio por muestra
        avg_mae  : MAE promedio por muestra (en °C si targets están en °C)
    """
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss = 0.0
    total_mae  = 0.0
    n          = 0

    with torch.set_grad_enabled(training):
        for inputs, targets in loader:
            inputs  = inputs.to(device)   # (batch, 120, 14)
            targets = targets.to(device)  # (batch,)  en °C

            preds = model(inputs)         # (batch,)  en °C (targets no normalizados)
            loss  = criterion(preds, targets)

            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            batch_n     = inputs.size(0)
            total_loss += loss.item() * batch_n
            total_mae  += torch.sum(torch.abs(preds - targets)).item()
            n          += batch_n

    return total_loss / n, total_mae / n


def get_predictions(model, dataset):
    """
    Devuelve predicciones y targets en °C para visualización.

    Nota: la red predice en la escala normalizada, aquí desnormalizamos.
    Los targets ya están en °C (temperature array de la Parte 1).
    """
    model.eval()
    all_preds   = []
    all_targets = []
    loader = DataLoader(dataset, batch_size=256, shuffle=False)

    with torch.no_grad():
        for inputs, targets in loader:
            preds = model(inputs.to(device)).cpu().numpy()
            # Desnormalizar predicciones: ŷ_real = ŷ_norm × σ + μ
            all_preds.append(preds * temp_std + temp_mean)
            all_targets.append(targets.numpy())  # ya en °C

    return np.concatenate(all_preds), np.concatenate(all_targets)


def plot_training_curves(history, model_name, baseline_mae):
    """Grafica curvas de loss y MAE a lo largo de las épocas."""
    epochs = range(1, len(history['train_loss']) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Loss
    axes[0].plot(epochs, history['train_loss'], label='Train loss', color='steelblue')
    axes[0].plot(epochs, history['val_loss'],   label='Val loss',   color='tomato')
    axes[0].set_title(f'{model_name} — Loss (MSE)')
    axes[0].set_xlabel('Época')
    axes[0].set_ylabel('MSE')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    # MAE
    axes[1].plot(epochs, history['train_mae'], label='Train MAE', color='steelblue')
    axes[1].plot(epochs, history['val_mae'],   label='Val MAE',   color='tomato')
    axes[1].axhline(baseline_mae, color='gray', linestyle='--', linewidth=1.2,
                    label=f'Baseline naive ({baseline_mae:.2f}°C)')
    axes[1].set_title(f'{model_name} — MAE (°C)')
    axes[1].set_xlabel('Época')
    axes[1].set_ylabel('MAE (°C)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(f'Curvas de entrenamiento — {model_name}', fontsize=13)
    plt.tight_layout()
    plt.show()


def plot_predictions(preds, targets, model_name, n_show=500):
    """Grafica predicciones vs. valores reales sobre los primeros n_show puntos del test."""
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(targets[:n_show], label='Real (°C)',       color='steelblue', linewidth=0.9)
    ax.plot(preds[:n_show],   label='Predicción (°C)', color='tomato',    linewidth=0.9, alpha=0.8)
    mae = np.mean(np.abs(preds[:n_show] - targets[:n_show]))
    ax.set_title(f'{model_name} — Predicciones vs. Real (test, primeros {n_show} puntos)  |  MAE={mae:.2f}°C')
    ax.set_xlabel('Muestra')
    ax.set_ylabel('Temperatura (°C)')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


print('✅ Funciones definidas: run_epoch | get_predictions | plot_training_curves | plot_predictions')

---
## Celda 8 — Entrenar SimpleRNN

Entrenamos el modelo básico con:
- `ModelCheckpoint` → guarda el mejor modelo según `val_mae`
- `EarlyStopping` → para si no mejora en 7 épocas
- `ReduceLROnPlateau` → baja el LR si se estanca


In [ ]:
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando dispositivo: {device}')

# ── Setup ──────────────────────────────────────────────────────────────────────
model_simple  = SimpleRNNModel(num_features=num_features, hidden_size=16).to(device)
optimizer     = torch.optim.Adam(model_simple.parameters(), lr=1e-3)
criterion     = nn.MSELoss()
writer        = SummaryWriter(log_dir='runs/jena_simple_rnn')

checkpoint_simple = ModelCheckpoint('jena_simple_rnn_best.pt', monitor='val_mae', verbose=True)
early_stop        = EarlyStopping(monitor='val_mae', patience=7)
reduce_lr         = ReduceLROnPlateau(optimizer, monitor='val_mae', patience=3, factor=0.5)

# ── Historial para graficar curvas ─────────────────────────────────────────────
history_simple = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}

# ── Bucle de entrenamiento ─────────────────────────────────────────────────────
epochs = 20
print(f'\nEntrenando SimpleRNN por hasta {epochs} épocas...')
print('-' * 75)

for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model_simple, train_loader, criterion, optimizer)
    val_loss,   val_mae   = run_epoch(model_simple, val_loader,   criterion)

    metrics = {'train_loss': train_loss, 'train_mae': train_mae,
               'val_loss':   val_loss,   'val_mae':   val_mae}

    # Guardar historial
    for k, v in metrics.items():
        history_simple[k].append(v)

    # TensorBoard
    writer.add_scalars('Loss_simple', {'train': train_loss, 'val': val_loss}, epoch)
    writer.add_scalars('MAE_simple',  {'train': train_mae,  'val': val_mae},  epoch)

    print(f'Época {epoch:02d}/{epochs} — '
          f'loss: {train_loss:.4f} | MAE: {train_mae:.4f}°C  ||  '
          f'val_loss: {val_loss:.4f} | val_MAE: {val_mae:.4f}°C')

    # Callbacks
    checkpoint_simple.step(metrics, model_simple)
    reduce_lr.step(metrics)
    early_stop.step(metrics)
    if early_stop.should_stop:
        break

writer.close()
print('-' * 75)
print('✅ Entrenamiento SimpleRNN terminado')

---
## Celda 9 — Evaluar y visualizar SimpleRNN


In [ ]:
# Cargar mejor modelo guardado por el checkpoint
model_simple.load_state_dict(torch.load('jena_simple_rnn_best.pt', map_location=device))

_, test_mae_simple = run_epoch(model_simple, test_loader, criterion)
print(f'SimpleRNN — Test MAE : {test_mae_simple:.4f}°C')
print(f'Baseline naive       : {baseline_mae:.4f}°C')
if test_mae_simple < baseline_mae:
    print(f'✅ Supera el baseline por {baseline_mae - test_mae_simple:.4f}°C')
else:
    print(f'❌ No supera el baseline (diferencia: {test_mae_simple - baseline_mae:.4f}°C)')

# Curvas de entrenamiento
plot_training_curves(history_simple, 'SimpleRNN', baseline_mae)

# Predicciones vs real
preds_simple, targets_simple = get_predictions(model_simple, test_dataset)
plot_predictions(preds_simple, targets_simple, 'SimpleRNN')

---
## Concepto: ¿Qué es una RNN apilada (Stacked)?

En una RNN apilada, la **salida de la capa 1** (la secuencia completa de estados ocultos) se convierte en la **entrada de la capa 2**.

```
Entrada original:
  x[0]  x[1]  ...  x[119]     (14 features cada uno)
   ↓     ↓           ↓
[RNN Capa 1]
   ↓     ↓           ↓
  h1[0] h1[1] ... h1[119]    (16 estados ocultos cada uno)
   ↓     ↓           ↓
[RNN Capa 2]
   ↓     ↓           ↓
  h2[0] h2[1] ... h2[119]
                     ↓
                [Linear] → ŷ
```

La capa 2 aprende **patrones de más alto nivel** sobre los patrones que ya aprendió la capa 1.
Es análogo a las CNNs: capas tempranas detectan bordes, capas profundas detectan formas.

**Más capas = más capacidad**, pero también más riesgo de overfitting y más lento de entrenar.


---
## Celda 10 — Entrenar StackedRNN


In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────────
model_stacked  = StackedRNNModel(num_features=num_features, hidden_size=16).to(device)
optimizer_s    = torch.optim.Adam(model_stacked.parameters(), lr=1e-3)
writer_s       = SummaryWriter(log_dir='runs/jena_stacked_rnn')

checkpoint_stacked = ModelCheckpoint('jena_stacked_rnn_best.pt', monitor='val_mae', verbose=True)
early_stop_s       = EarlyStopping(monitor='val_mae', patience=7)
reduce_lr_s        = ReduceLROnPlateau(optimizer_s, monitor='val_mae', patience=3, factor=0.5)

history_stacked = {'train_loss': [], 'val_loss': [], 'train_mae': [], 'val_mae': []}

# ── Bucle de entrenamiento ─────────────────────────────────────────────────────
print(f'\nEntrenando StackedRNN por hasta {epochs} épocas...')
print('-' * 75)

for epoch in range(1, epochs + 1):
    train_loss, train_mae = run_epoch(model_stacked, train_loader, criterion, optimizer_s)
    val_loss,   val_mae   = run_epoch(model_stacked, val_loader,   criterion)

    metrics = {'train_loss': train_loss, 'train_mae': train_mae,
               'val_loss':   val_loss,   'val_mae':   val_mae}

    for k, v in metrics.items():
        history_stacked[k].append(v)

    writer_s.add_scalars('Loss_stacked', {'train': train_loss, 'val': val_loss}, epoch)
    writer_s.add_scalars('MAE_stacked',  {'train': train_mae,  'val': val_mae},  epoch)

    print(f'Época {epoch:02d}/{epochs} — '
          f'loss: {train_loss:.4f} | MAE: {train_mae:.4f}°C  ||  '
          f'val_loss: {val_loss:.4f} | val_MAE: {val_mae:.4f}°C')

    checkpoint_stacked.step(metrics, model_stacked)
    reduce_lr_s.step(metrics)
    early_stop_s.step(metrics)
    if early_stop_s.should_stop:
        break

writer_s.close()
print('-' * 75)
print('✅ Entrenamiento StackedRNN terminado')

---
## Celda 11 — Evaluar y visualizar StackedRNN


In [ ]:
model_stacked.load_state_dict(torch.load('jena_stacked_rnn_best.pt', map_location=device))

_, test_mae_stacked = run_epoch(model_stacked, test_loader, criterion)
print(f'StackedRNN — Test MAE : {test_mae_stacked:.4f}°C')
print(f'Baseline naive        : {baseline_mae:.4f}°C')
if test_mae_stacked < baseline_mae:
    print(f'✅ Supera el baseline por {baseline_mae - test_mae_stacked:.4f}°C')
else:
    print(f'❌ No supera el baseline (diferencia: {test_mae_stacked - baseline_mae:.4f}°C)')

plot_training_curves(history_stacked, 'StackedRNN', baseline_mae)

preds_stacked, targets_stacked = get_predictions(model_stacked, test_dataset)
plot_predictions(preds_stacked, targets_stacked, 'StackedRNN')

---
## Celda 12 — Tabla comparativa final

Comparamos los tres modelos: **Naive Baseline**, **SimpleRNN** y **StackedRNN**.


In [ ]:
results = [
    ('Naive Baseline',  baseline_mae,       '—',  '—'),
    ('SimpleRNN',       test_mae_simple,     '1',  '16'),
    ('StackedRNN',      test_mae_stacked,    '2',  '16'),
]

print('╔══════════════════════════════════════════════════════════════╗')
print('║              TABLA COMPARATIVA FINAL                        ║')
print('╠══════════════════╦═══════════╦══════════╦════════════════════╣')
print('║ Modelo           ║  Test MAE ║  Capas   ║  hidden_size       ║')
print('╠══════════════════╬═══════════╬══════════╬════════════════════╣')
for name, mae, layers, hidden in results:
    print(f'║ {name:<16s} ║  {mae:>6.4f}°C ║  {layers:<6s}  ║  {hidden:<18s} ║')
print('╚══════════════════╩═══════════╩══════════╩════════════════════╝')

best_model = min(results[1:], key=lambda x: x[1])
print(f'\n🥇 Mejor modelo: {best_model[0]} — MAE {best_model[1]:.4f}°C')
print(f'   Mejora vs Naive: {baseline_mae - best_model[1]:.4f}°C ({(1 - best_model[1]/baseline_mae)*100:.1f}% mejor)')

# ── Gráfica de barras comparativa ─────────────────────────────────────────────
nombres = [r[0] for r in results]
maes    = [r[1] for r in results]
colores = ['#aaaaaa', 'steelblue', 'tomato']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(nombres, maes, color=colores, edgecolor='black', linewidth=0.8)

for bar, mae in zip(bars, maes):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{mae:.4f}°C', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.axhline(baseline_mae, color='gray', linestyle='--', linewidth=1.2)
ax.set_title('Test MAE (°C) — menor es mejor', fontsize=13)
ax.set_ylabel('MAE (°C)')
ax.set_ylim(0, max(maes) * 1.2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Notas para seguir experimentando

| Idea | Cómo hacerlo |
|---|---|
| Más capacidad | Subir `hidden_size` a 32, 64 o 128 |
| Regularización | Añadir `dropout=0.2` a `StackedRNNModel` |
| Secuencias más largas | Cambiar `sequence_length` a 240 (10 días) |
| Arquitectura LSTM | Reemplazar `nn.RNN` por `nn.LSTM` (misma API) |
| Arquitectura GRU | Reemplazar `nn.RNN` por `nn.GRU` (misma API) |
| LR más agresivo | Probar `lr=1e-2` y ver si converge más rápido |

**La transición de `nn.RNN` a `nn.LSTM` o `nn.GRU` es literalmente cambiar una sola palabra en el código.**
El resto del notebook funciona igual — ese es el poder de la API de PyTorch.
